# Семинар 2. Исследование методов линейной регрессии

#### Критерий оценивания:
#### Пункт 14: максимум 0,5 балла
#### Пункт 15: максимум 0,5 балла

#### Итого за работу: максимум 1 балл
#### P.S. пункты 14 и 15 без пунктов 1-13 не засчитываются, ответы на вопросы из ИИ не засчитываются

1. Загрузите датасет и выведите на экран первые несколько строк

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

data = pd.read_csv("auto_dataset.csv")
display(data.head())
print("Размер:", data.shape)
print("Пропуски:", data.isna().sum().sum())

Размер: (1000, 10)
Пропуски: 0


,brand,model,vehicleType,gearbox,fuelType,notRepairedDamage,powerPS,kilometer,autoAgeMonths,price
0,volkswagen,golf,kleinwagen,manuell,benzin,nein,75,150000,177,1500
1,skoda,fabia,kleinwagen,manuell,diesel,nein,69,90000,93,3600
2,bmw,3er,limousine,manuell,benzin,ja,102,150000,246,650
3,peugeot,2_reihe,cabrio,manuell,benzin,nein,109,150000,140,2200
4,mazda,3_reihe,limousine,manuell,benzin,nein,105,150000,136,2000


2. Разбейте выборку на признаки и ответы. Закодируйте категориальные признаки.

In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

X = data.drop(columns="price")
y = data["price"]
cat_cols = X.select_dtypes(include="object").columns.tolist()
num_cols = X.select_dtypes(include="number").columns.tolist()

# Кодировщик и масштабирование обучим только на train после разбиения.
preprocessor = ColumnTransformer([
    ("numbers", StandardScaler(), num_cols),
    ("categories", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)
])

3. Разбейте датасет на train val test в отношении 8:1:1

In [3]:
X_train_raw, X_other, y_train, y_other = train_test_split(
    X, y, test_size=0.2, random_state=42)
X_val_raw, X_test_raw, y_val, y_test = train_test_split(
    X_other, y_other, test_size=0.5, random_state=42)

X_train = preprocessor.fit_transform(X_train_raw)
X_val = preprocessor.transform(X_val_raw)
X_test = preprocessor.transform(X_test_raw)
X_train = np.column_stack([np.ones(len(X_train)), X_train])
X_val = np.column_stack([np.ones(len(X_val)), X_val])
X_test = np.column_stack([np.ones(len(X_test)), X_test])
y_train = y_train.to_numpy(dtype=float)
y_val = y_val.to_numpy(dtype=float)
y_test = y_test.to_numpy(dtype=float)

y_mean = y_train.mean()
y_scale = y_train.std()
y_scaled = (y_train - y_mean) / y_scale
print("Размеры выборок:", len(y_train), len(y_val), len(y_test))
print("Число признаков со свободным членом:", X_train.shape[1])

Размеры выборок: 800 100 100
Число признаков со свободным членом: 195


4. Исследуйте VGD с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите VGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте VGD на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте VGD c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

Loss — это MSE в исходных единицах цены в квадрате. Для обучения масштабируем числовые признаки и цену по train. Перебираем шаги от $10^{-5}$ до 1; всем методам даём 5000 обновлений. SGD использует батч из 32 автомобилей, SAG — один объект. Остальные параметры берём из лекции. Явно расходящиеся запуски останавливаем и не выбираем.

На test модель не обучается, поэтому число итераций **на test равно 0**. Отдельно указываем число итераций обучения выбранной модели. Тестовые прогнозы считаем один раз после выбора шага по val.

In [4]:
steps = np.logspace(-5, 0, 6)
max_iter = 5000


def fit(method, lyamda, decay):
    X = X_train
    y = y_scaled
    n, d = X.shape
    w = np.zeros(d)
    rng = np.random.default_rng(42)
    momentum = np.zeros(d)
    m = np.zeros(d)
    v = np.zeros(d)
    memory = np.zeros((n, d))
    avg_grad = np.zeros(d)
    train_history = []
    val_history = []

    for k in range(max_iter):
        eta = lyamda / np.sqrt(k + 1) if decay else lyamda
        if method == "StochasticGradientDescent":
            batch = rng.choice(n, size=32, replace=False)
            grad = 2 * X[batch].T @ (X[batch] @ w - y[batch]) / len(batch)
        elif method == "SAGDescent":
            j = rng.integers(n)
            new_grad = 2 * (X[j] @ w - y[j]) * X[j]
            avg_grad += (new_grad - memory[j]) / n
            memory[j] = new_grad
            grad = avg_grad
        else:
            grad = 2 * X.T @ (X @ w - y) / n

        if method == "MomentumDescent":
            momentum = 0.9 * momentum + eta * grad
            w -= momentum
        elif method == "Adam":
            m = 0.9 * m + 0.1 * grad
            v = 0.999 * v + 0.001 * grad ** 2
            m_hat = m / (1 - 0.9 ** (k + 1))
            v_hat = v / (1 - 0.999 ** (k + 1))
            w -= eta * m_hat / (np.sqrt(v_hat) + 1e-8)
        else:
            w -= eta * grad

        pred_train = (X_train @ w) * y_scale + y_mean
        pred_val = (X_val @ w) * y_scale + y_mean
        train_history.append(np.mean((pred_train - y_train) ** 2))
        val_history.append(np.mean((pred_val - y_val) ** 2))
        # Прерываем явно расходящийся запуск до переполнения чисел.
        if not np.isfinite(train_history[-1]) or train_history[-1] > 1e20:
            return {"w": w, "train_history": train_history,
                    "val_history": val_history, "converged": False}

    return {"w": w, "train_history": train_history, "val_history": val_history, "converged": True}


def predict(model, X):
    return (X @ model["w"]) * y_scale + y_mean



results = []
search_tables = {}


def compare(method, decay):
    rows = []
    models = []
    for step in steps:
        model = fit(method, step, decay)
        models.append(model)
        rows.append({
            "Шаг": step,
            "R2 train": r2_score(y_train, predict(model, X_train)),
            "Loss train": model["train_history"][-1],
            "Loss val": model["val_history"][-1],
            "Итерации": len(model["train_history"]),
            "Статус": "готово" if model["converged"] else "расходится"
        })
    table = pd.DataFrame(rows)
    display(table)
    best_index = table.loc[table["Статус"] == "готово", "Loss val"].idxmin()
    best = models[best_index]
    best_step = table.loc[best_index, "Шаг"]
    name = method + (" (TimeDecayLR)" if decay else " (постоянный)")
    search_tables[name] = table
    prediction = predict(best, X_test)
    result = {
        "Метод": name,
        "Лучший шаг": f"{best_step:g} / sqrt(k + 1)" if decay else f"{best_step:g}",
        "Loss_train": best["train_history"][-1],
        "Loss_val": best["val_history"][-1],
        "Loss_test": mean_squared_error(y_test, prediction),
        "R2_train": r2_score(y_train, predict(best, X_train)),
        "R2_test": r2_score(y_test, prediction),
        "Итерации обучения": len(best["train_history"]),
        "Итерации на test": 0
    }
    results.append(result)
    display(pd.DataFrame([result]))

compare("VanillaGradientDescent", decay=False)

,Шаг,R2 train,Loss train,Loss val,Итерации,Статус
0,0.00001,1.591928e-01,5.117356e+07,4.669157e+07,5000,готово
1,0.00010,6.134120e-01,2.352868e+07,2.265008e+07,5000,готово
2,0.00100,7.008792e-01,1.820521e+07,2.059038e+07,5000,готово
3,0.01000,7.503678e-01,1.519322e+07,1.825524e+07,5000,готово
4,0.10000,7.925173e-01,1.262790e+07,1.670697e+07,5000,готово
5,1.00000,-3.088275e+12,1.879599e+20,1.762828e+20,10,расходится


,Метод,Лучший шаг,Loss_train,Loss_val,Loss_test,R2_train,R2_test,Итерации обучения,Итерации на test
0,VanillaGradientDescent (постоянный),0.1,1.262790e+07,1.670697e+07,2.783547e+07,0.792517,0.600854,5000,0


5. Исследуйте VGD с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите VGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте VGD на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте VGD c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [5]:
compare("VanillaGradientDescent", decay=True)

,Шаг,R2 train,Loss train,Loss val,Итерации,Статус
0,0.00001,0.005106,6.055166e+07,5.508580e+07,5000,готово
1,0.00010,0.049256,5.786460e+07,5.267982e+07,5000,готово
2,0.00100,0.352961,3.938035e+07,3.617655e+07,5000,готово
3,0.01000,0.674618,1.980354e+07,2.090665e+07,5000,готово
4,0.10000,0.719403,1.707780e+07,1.984588e+07,5000,готово
5,1.00000,0.774341,1.373414e+07,1.707195e+07,5000,готово


,Метод,Лучший шаг,Loss_train,Loss_val,Loss_test,R2_train,R2_test,Итерации обучения,Итерации на test
0,VanillaGradientDescent (TimeDecayLR),1 / sqrt(k + 1),1.373414e+07,1.707195e+07,2.645872e+07,0.774341,0.620595,5000,0


6. Исследуйте SGD с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите SGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SGD на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SGD c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [6]:
compare("StochasticGradientDescent", decay=False)

,Шаг,R2 train,Loss train,Loss val,Итерации,Статус
0,0.00001,1.589948e-01,5.118561e+07,4.670027e+07,5000,готово
1,0.00010,6.136506e-01,2.351416e+07,2.263870e+07,5000,готово
2,0.00100,7.012328e-01,1.818370e+07,2.060551e+07,5000,готово
3,0.01000,7.504246e-01,1.518977e+07,1.836379e+07,5000,готово
4,0.10000,7.834324e-01,1.318083e+07,1.709373e+07,5000,готово
5,1.00000,-2.305687e+12,1.403297e+20,1.390697e+20,9,расходится


,Метод,Лучший шаг,Loss_train,Loss_val,Loss_test,R2_train,R2_test,Итерации обучения,Итерации на test
0,StochasticGradientDescent (постоянный),0.1,1.318083e+07,1.709373e+07,2.827626e+07,0.783432,0.594533,5000,0


7. Исследуйте SGD с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите SGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SGD на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SGD c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [7]:
compare("StochasticGradientDescent", decay=True)

,Шаг,R2 train,Loss train,Loss val,Итерации,Статус
0,0.00001,0.005146,6.054922e+07,5.508346e+07,5000,готово
1,0.00010,0.049632,5.784169e+07,5.265820e+07,5000,готово
2,0.00100,0.354904,3.926207e+07,3.607936e+07,5000,готово
3,0.01000,0.675501,1.974978e+07,2.094329e+07,5000,готово
4,0.10000,0.719558,1.706838e+07,1.990830e+07,5000,готово
5,1.00000,-1.117997,1.289064e+08,1.584405e+08,5000,готово


,Метод,Лучший шаг,Loss_train,Loss_val,Loss_test,R2_train,R2_test,Итерации обучения,Итерации на test
0,StochasticGradientDescent (TimeDecayLR),0.1 / sqrt(k + 1),1.706838e+07,1.990830e+07,2.468293e+07,0.719558,0.646059,5000,0


8. Исследуйте SAG с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите SAG на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SAG на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SAG c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [8]:
compare("SAGDescent", decay=False)

,Шаг,R2 train,Loss train,Loss val,Итерации,Статус
0,0.00001,1.372386e-01,5.250975e+07,4.788423e+07,5000,готово
1,0.00010,6.141740e-01,2.348230e+07,2.243031e+07,5000,готово
2,0.00100,6.983191e-01,1.836103e+07,2.099830e+07,5000,готово
3,0.01000,7.345282e-01,1.615726e+07,1.749073e+07,5000,готово
4,0.10000,-3.958449e+01,2.470070e+09,2.969588e+09,5000,готово
5,1.00000,-1.705769e+12,1.038172e+20,1.038089e+20,2215,расходится


,Метод,Лучший шаг,Loss_train,Loss_val,Loss_test,R2_train,R2_test,Итерации обучения,Итерации на test
0,SAGDescent (постоянный),0.01,1.615726e+07,1.749073e+07,2.624674e+07,0.734528,0.623635,5000,0


9. Исследуйте SAG с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите SAG на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SAG на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SAG c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [9]:
compare("SAGDescent", decay=True)

,Шаг,R2 train,Loss train,Loss val,Итерации,Статус
0,0.00001,0.003284,6.066254e+07,5.518508e+07,5000,готово
1,0.00010,0.032176,5.890409e+07,5.361047e+07,5000,готово
2,0.00100,0.263690,4.481361e+07,4.100018e+07,5000,готово
3,0.01000,0.664807,2.040063e+07,2.115376e+07,5000,готово
4,0.10000,0.709942,1.765363e+07,2.039529e+07,5000,готово
5,1.00000,0.741538,1.573065e+07,2.000369e+07,5000,готово


,Метод,Лучший шаг,Loss_train,Loss_val,Loss_test,R2_train,R2_test,Итерации обучения,Итерации на test
0,SAGDescent (TimeDecayLR),1 / sqrt(k + 1),1.573065e+07,2.000369e+07,2.747637e+07,0.741538,0.606003,5000,0


10. Исследуйте Momentum с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите Momentum на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Momentum на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Momentum c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [10]:
compare("MomentumDescent", decay=False)

,Шаг,R2 train,Loss train,Loss val,Итерации,Статус
0,0.00001,6.134593e-01,2.352580e+07,2.264649e+07,5000,готово
1,0.00010,7.008656e-01,1.820604e+07,2.059200e+07,5000,готово
2,0.00100,7.503464e-01,1.519453e+07,1.825630e+07,5000,готово
3,0.01000,7.925150e-01,1.262804e+07,1.670711e+07,5000,готово
4,0.10000,8.021865e-01,1.203941e+07,1.680464e+07,5000,готово
5,1.00000,-1.648933e+12,1.003580e+20,9.412850e+19,11,расходится


,Метод,Лучший шаг,Loss_train,Loss_val,Loss_test,R2_train,R2_test,Итерации обучения,Итерации на test
0,MomentumDescent (постоянный),0.01,1.262804e+07,1.670711e+07,2.783608e+07,0.792515,0.600845,5000,0


11. Исследуйте Momentum с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите Momentum на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Momentum на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Momentum c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [11]:
compare("MomentumDescent", decay=True)

,Шаг,R2 train,Loss train,Loss val,Итерации,Статус
0,0.00001,0.049219,5.786683e+07,5.268181e+07,5000,готово
1,0.00010,0.353061,3.937425e+07,3.617088e+07,5000,готово
2,0.00100,0.674671,1.980031e+07,2.091152e+07,5000,готово
3,0.01000,0.719408,1.707753e+07,1.984668e+07,5000,готово
4,0.10000,0.774371,1.373231e+07,1.706890e+07,5000,готово
5,1.00000,0.799262,1.221737e+07,1.677847e+07,5000,готово


,Метод,Лучший шаг,Loss_train,Loss_val,Loss_test,R2_train,R2_test,Итерации обучения,Итерации на test
0,MomentumDescent (TimeDecayLR),1 / sqrt(k + 1),1.221737e+07,1.677847e+07,2.894401e+07,0.799262,0.584958,5000,0


12. Исследуйте Adam с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите Adam на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Adam на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Adam c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [12]:
compare("Adam", decay=False)

,Шаг,R2 train,Loss train,Loss val,Итерации,Статус
0,0.00001,0.262777,4.486920e+07,4.061327e+07,5000,готово
1,0.00010,0.758742,1.468352e+07,1.672170e+07,5000,готово
2,0.00100,0.801938,1.205451e+07,1.742901e+07,5000,готово
3,0.01000,0.802369,1.202827e+07,1.779318e+07,5000,готово
4,0.10000,0.802367,1.202843e+07,1.795237e+07,5000,готово
5,1.00000,0.802266,1.203458e+07,1.882539e+07,5000,готово


,Метод,Лучший шаг,Loss_train,Loss_val,Loss_test,R2_train,R2_test,Итерации обучения,Итерации на test
0,Adam (постоянный),0.0001,1.468352e+07,1.672170e+07,2.713031e+07,0.758742,0.610965,5000,0


13. Исследуйте Adam с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите Adam на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Adam на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Adam c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [13]:
compare("Adam", decay=True)

,Шаг,R2 train,Loss train,Loss val,Итерации,Статус
0,0.00001,0.008678,6.033426e+07,5.486127e+07,5000,готово
1,0.00010,0.083373,5.578813e+07,5.061315e+07,5000,готово
2,0.00100,0.544949,2.769551e+07,2.577876e+07,5000,готово
3,0.01000,0.795366,1.245450e+07,1.758104e+07,5000,готово
4,0.10000,0.802369,1.202827e+07,1.800336e+07,5000,готово
5,1.00000,0.802369,1.202827e+07,1.851860e+07,5000,готово


,Метод,Лучший шаг,Loss_train,Loss_val,Loss_test,R2_train,R2_test,Итерации обучения,Итерации на test
0,Adam (TimeDecayLR),0.01 / sqrt(k + 1),1.245450e+07,1.758104e+07,2.839814e+07,0.795366,0.592785,5000,0


14. Постройте итоговую сравнительную таблицу со следующими столбцами:

1) название метода

2) значение лучшего шага (если n) или функция лучшего шага (если n(lyamda))

3) Loss_train

4) Loss_test

5) R^2 train

6) R^2 test

7) число итераций на test

In [14]:
comparison = pd.DataFrame(results)
pd.set_option("display.max_columns", None)
display(comparison)

,Метод,Лучший шаг,Loss_train,Loss_val,Loss_test,R2_train,R2_test,Итерации обучения,Итерации на test
0,VanillaGradientDescent (постоянный),0.1,1.262790e+07,1.670697e+07,2.783547e+07,0.792517,0.600854,5000,0
1,VanillaGradientDescent (TimeDecayLR),1 / sqrt(k + 1),1.373414e+07,1.707195e+07,2.645872e+07,0.774341,0.620595,5000,0
2,StochasticGradientDescent (постоянный),0.1,1.318083e+07,1.709373e+07,2.827626e+07,0.783432,0.594533,5000,0
3,StochasticGradientDescent (TimeDecayLR),0.1 / sqrt(k + 1),1.706838e+07,1.990830e+07,2.468293e+07,0.719558,0.646059,5000,0
4,SAGDescent (постоянный),0.01,1.615726e+07,1.749073e+07,2.624674e+07,0.734528,0.623635,5000,0
5,SAGDescent (TimeDecayLR),1 / sqrt(k + 1),1.573065e+07,2.000369e+07,2.747637e+07,0.741538,0.606003,5000,0
6,MomentumDescent (постоянный),0.01,1.262804e+07,1.670711e+07,2.783608e+07,0.792515,0.600845,5000,0
7,MomentumDescent (TimeDecayLR),1 / sqrt(k + 1),1.221737e+07,1.677847e+07,2.894401e+07,0.799262,0.584958,5000,0
8,Adam (постоянный),0.0001,1.468352e+07,1.672170e+07,2.713031e+07,0.758742,0.610965,5000,0
9,Adam (TimeDecayLR),0.01 / sqrt(k + 1),1.245450e+07,1.758104e+07,2.839814e+07,0.795366,0.592785,5000,0


15. Сделайте вывод о том, какой метод и шаг линейной регрессии самый лучший для данной выборки и ответьте на вопросы:

1) почему именно этот метод и этот шаг самый лучший (по каким данным из таблицы вы сделали такой вывод)

2) расскажите простыми словами суть R^2_train и R^2_test?

3) как R^2_train и R^2_test помогают сравнивать методы? почему оба эти значения надо вычислять для данного выбора?

In [15]:
best_test = comparison.loc[comparison["Loss_test"].idxmin()]
best_val = comparison.loc[comparison["Loss_val"].idxmin()]
print("Лучшее качество на test:", best_test["Метод"], ", шаг:", best_test["Лучший шаг"])
print(f"Loss_test = {best_test['Loss_test']:.2f}, R2_test = {best_test['R2_test']:.3f}")
print("Выбор по val:", best_val["Метод"], ", шаг:", best_val["Лучший шаг"])

Лучшее качество на test: StochasticGradientDescent (TimeDecayLR) , шаг: 0.1 / sqrt(k + 1)
Loss_test = 24682925.01, R2_test = 0.646
Выбор по val: VanillaGradientDescent (постоянный) , шаг: 0.1


Лучшим методом по валидационной выборке получился VGD с постоянным шагом 0,1, потому что у него самая маленькая ошибка Loss val. Но на тестовой выборке лучше всего SGD с уменьшающимся шагом 0,1 / (k + 1)**0,5. У него самая маленькая Loss test и самый большой R2 test.

R2 train показывает, насколько хорошо модель описала данные, на которых она училась. R2 test показывает, насколько хорошо она предсказывает новые данные, которые она не видела. Если R2 близок к 1, значит модель хорошо предсказывает. Если R2 близок к 0, значит модель почти не лучше, чем просто предсказывать среднее значение.

Нужно смотреть и R2 train, и R2 test. Если на train R2 высокий, а на test намного ниже, значит модель могла переобучиться. То есть она хорошо запомнила тренировочные данные, но хуже работает на новых.